In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                             confusion_matrix, classification_report)
from utils import load_data

X_train, X_test, y_train, y_test = load_data()

In [ ]:


# 1. Preprocessing — One-Hot Encoding 
# Linear regression interprets all inputs as continuous numbers.
# If we label-encode Sales=1, HR=2, Tech=3, the model assumes
# Tech > HR > Sales numerically, which is meaningless.
# One-hot encoding avoids this by making each category a binary column.

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=categorical_cols, drop_first=True)

# Align columns so that test set might have fewer dummy columns if a category
# was absent in the test split
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join="left", axis=1, fill_value=0)

print("Shape after encoding:", X_train_enc.shape)

#  2. Preprocessing — Feature Scaling 
# Linear regression finds weights via gradient descent or closed-form
# solution. Features on different scales cause some weights to dominate
#just because of their magnitude and not importance.
# MonthlyIncome: 0–20,000 vs JobSatisfaction: 1–4
# Without scaling, the model overweights income not because it matters
# more, but because its numbers are bigger.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)  # fit only on train
X_test_scaled  = scaler.transform(X_test_enc)       # only transform test

# 3. Train the model 
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# 4. Raw predictions
y_pred_raw = model.predict(X_test_scaled)

print("\n── Raw prediction samples ──")
print(y_pred_raw[:10].round(3))
# You'll see values like 0.31, -0.08, 0.77, 1.12 ...
# Negative values and values > 1 are nonsensical for a 0/1 target
# This is the fundamental problem with Linear Regression for classification

print(f"\nMin prediction : {y_pred_raw.min():.3f}")
print(f"Max prediction : {y_pred_raw.max():.3f}")
print(f"Values below 0 : {(y_pred_raw < 0).sum()}")
print(f"Values above 1 : {(y_pred_raw > 1).sum()}")

#  5. Force it into classification by thresholding 
# Since we need 0/1 predictions for comparison, we apply a 0.5 cutoff.
y_pred_class = (y_pred_raw >= 0.5).astype(int)

# 6. Evaluate 
print("\n── Regression metrics (don't mean much here) ──")
print(f"MSE : {mean_squared_error(y_test, y_pred_raw):.4f}")
print(f"R²  : {r2_score(y_test, y_pred_raw):.4f}")

print("\n── Classification metrics (after 0.5 threshold) ──")
print(classification_report(y_test, y_pred_class, target_names=["No", "Yes"]))

cm = confusion_matrix(y_test, y_pred_class)
print("Confusion Matrix:\n", cm)



C:\Users\shrad\AppData\Local\Temp\ipykernel_17636\3194879123.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


Shape after encoding: (1176, 44)

── Raw prediction samples ──
[ 0.206 -0.216  0.234 -0.091  0.299  0.312 -0.014  0.073 -0.12   0.196]

Min prediction : -0.293
Max prediction : 0.842
Values below 0 : 64
Values above 1 : 0

── Regression metrics (don't mean much here) ──
MSE : 0.1128
R²  : 0.1600

── Classification metrics (after 0.5 threshold) ──
              precision    recall  f1-score   support

          No       0.87      0.99      0.92       247
         Yes       0.77      0.21      0.33        47

    accuracy                           0.86       294
   macro avg       0.82      0.60      0.63       294
weighted avg       0.85      0.86      0.83       294

Confusion Matrix:
 [[244   3]
 [ 37  10]]
